## Tutorial 2 - Building the system

Now we have our fixed, oriented protein in Martini resolution, it is now time to embed this in a membrane! There are a growing number of useful and cool tools to build Martini membranes. We are going to use the simplest tool, which builds flat membranes, called [insane](https://pubs.acs.org/jctcce/article-abstract/11/5/2144/792662/Computational-Lipidomics-with-insane-A-Versatile?redirectedFrom=fulltext).  

If you are interested in building more complex systems, [MemPrO](https://pubs.acs.org/jctcce/article/22/1/638/5071691/MemPrO-A-Predictive-Tool-for-Membrane-Protein) builds upon insane to enable generation of curved membranes and double membrane systems. [COBY](https://pubs.acs.org/jcisd8/article/65/10/4760/3686743/Creating-Coarse-Grained-Systems-with-COBY-Toward) also enables the building of multiple membranes, as well as lipid patches, insertion of multiple proteins and different membrane shapes. [TS2CG2](https://pubs.acs.org/jctcce/article/21/18/9136/3693441/TS2CG-as-a-Membrane-Builder) enables the building of complex shapes from a triangular mesh, which is great for converting experimental data such as cryo-EM maps to a simulatable system. It also allows domains around proteins and the placement of multiple proteins based on placement rules. 

Back to Insane (for the membrane). Insane uses an (in Tsjerk Wassenaar's words, insanely) simple method of building lipids in a grid around the protein. We can see the options below for how we can tune our input:

In [ ]:
%%bash

##The command below might need to be altered slightly depending on what you named the outputs in the previous tutorial

cp ../tutorial_1/topol.top ../tutorial_1/VDAC1_cg.pdb ../tutorial_1/*.itp .

In [ ]:
!insane -h

Again, there are many options for the input here, but we will focus on the ones used in this tutorial:

- `-f` which specifies the input file for your CG protein. This will be a .gro or .pdb file
- `-o` which gives the output coordinate file a name. This will normally be a .gro file
- `-p` the name of the output topology file. This will be a .top file
- `-pbc` the type of periodic boundary to use. Square is the easiest in terms of analysis
- `-d` the distance between periodic images. We want to make sure this is large enough that the protein does not feel the effects of itself
- `-l` this is the lipid species on the _lower_ leaflet of the bilayer and is input like **POPE:10** and can be listed multiple times for each lipid of that bilayer. POPE is the lipid code found in insane (and the martini forcefield), and 10 would be the ratio (if it is the only thing specified it would be 100% of that leaflet, but if you also had POPG:90 there would be 10% POPE and 90% of POPG in the lower leaflet)
- `-u` is the lipid species on the _upper_ leaflet of the bilayer, and listing these differently allows bilayer asymmetry
- `-a` is the area per lipid. If you want to have a different area per lipid for the lower leaflet, `-au` would be for the upper leaflet and `-a` for the lower. It is good to use an area per lipid (APL) that is close to the value expected to reduce initial pressure changes
- `-au` is the area per lipid of the upper leaflet, and would be needed if an asymmetric bilayer is constructed
- `-center` ensures the protein is in the center of the bilayer, in terms of x and y
- `-sol` will add Martini water for us (which we will call `W`)
- `-salt` will add NaCl at a concentration set by the user; 0.15 nM is normally a good value for biological simulations

Try and use insane to build a simplified outer mitocondrial membrane bilayer of POPC:POPE:POPI (55:21:24) for the lower leaflet and POPC:POPE:POPI (56:40:4) for the upper leaflet (based from this [publication](https://pubs.acs.org/jcisd8/article-abstract/62/4/1036/982326/Comparative-Molecular-Dynamics-Simulation-Studies?redirectedFrom=fulltext)) as the composition. We can use an area per lipid of 0.58 nm<sup>2</sup> for the lower leaflet and 0.59 nm<sup>2</sup> for the upper leaflet. 

<details>
<summary>    
Interested in how to calculate area per lipid for your leaflet?
</summary>

This can be done by running a small symmetric bilayer of your desired composition and then calculating the area based on how many lipids you have in one leaflet. This is well worth doing if you are using a lipid composition that is different to that simulated before and you have an asymmetric membrane. More details can be found [here](https://cgmartini.nl/docs/tutorials/Martini3/LipidsI/#area-per-lipid).

</details>

In [ ]:
!insane -f VDAC1_cg.pdb ....

<details>
<summary>    
<i>Really</i> stuck? Click on this to reveal a command we can use
</summary>

`!insane -f VDAC1_cg.pdb -o system.gro -p topol.top -pbc square -d 8 -l POPC:55 -l POPE:21 -l POPI:24 -u POPC:56 -u POPE:40 -u POPI:4 -a 0.58 -au 0.59 -center -sol W -salt 0.15`  

If there's anything you don't understand, please ask!

</details>

We need to alter the topology file generated, as currently it does not point towards the .itp file we generated for our protein or relevant Martini force field files. The lines below replace the placeholder line `#include "martini.itp"` with paths to the actual files, and replace the general 'protein' with the name we defined within the `martinize2` command

In [ ]:
%%bash
sed -i 's|#include "martini.itp"|#include "itps/martini_v3.0.0.itp"\n#include "itps/martini_v3.0.0_ffbonded_v2.itp"\n#include "itps/martini_v3.0.0_ions_v1.itp"\n#include "itps/martini_v3.0.0_solvents_v1.itp"\n#include "itps/martini_v3.0.0_phospholipids_PC_v2.itp"\n#include "itps/martini_v3.0.0_phospholipids_PE_v2.itp"\n#include "itps/martini_v3.0.0_phospholipids_PI_v2.itp"\n#include "itps/VDAC1_0.itp"|' topol.top
sed -i 's/Protein/VDAC1_0/g' topol.top

Do this **out of the notebook** in your own terminal:

```$vmd your_CG_system.pdb```

Hopefully, from this visualisation you can see the grid-like nature in which `insane` builds lipids/bilayers. We now need to relax this system before we perform our production simulation, and we will look at energy minimisation in the next tutorial.